문항 1. 기존 MLP(Multi-Layer Perceptron) 모델은 이미지 처리에 근본적인 한계가 있습니다. 강의 자료에서 언급된 MLP의 한계점 2가지를 서술하시오.

답안:  

1. 공간 정보 손실

2. 파라미터 폭발



문항 2. CNN의 풀링(Pooling) 레이어는 크게 두 가지 주요 역할을 수행합니다. 이 두 가지 역할을 서술하시오.


답안:

1. 차원 축소

2. 수용 영역 확

문항 3. 데이터 전처리 정의 CIFAR-10 데이터셋을 불러오기 전, 이미지를 텐서로 변환하고 정규화(Normalize)하는 transform 객체를 정의하는 코드를 완성하시오. (조건: ToTensor()를 사용하고, 평균(mean)과 표준편차(std)는 모두 (0.5, 0.5, 0.5)로 설정하시오.)

In [3]:
import torch
from torchvision import transforms

# [문제] transform 정의 부분을 채우시오.
transform = transforms.Compose([
    # 1. 이미지를 PyTorch Tensor로 변환
    transforms.ToTensor(),
    #################### 빈칸 ####################,

    # 2. 정규화 (mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    #################### 빈칸 ####################
])

문항 4. CNN 모델 정의 - Feature Extractor SimpleCNN 클래스 내부의 self.features에 두 번째 합성곱 블록(Conv Block 2)을 추가하는 코드를 완성하시오.

(조건:

nn.Conv2d: 입력 채널 32, 출력 채널 64, 커널 크기 3, 패딩 1

nn.ReLU: 활성화 함수

nn.MaxPool2d: 커널 크기 2, 스트라이드 2)

In [4]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        # Feature Extractor 정의
        self.features = nn.Sequential(
            # Conv Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # [문제] Conv Block 2 코드를 아래에 완성하시오.
            nn.Conv2d(32,64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        # ... (Classifier 부분 생략)

문항 5. CNN 모델 정의 - Classifier self.features를 통과한 특징 맵의 크기는 [배치 크기, 64, 8, 8]입니다. 이 3차원 텐서를 1차원으로 펼치고(Flatten), nn.Linear 층에 연결하는 self.classifier 코드를 완성하시오.

(조건:

nn.Linear의 첫 번째 입력 크기(in_features)는 64 * 8 * 8 (즉, 4096)입니다.

nn.Linear의 출력 크기(out_features)는 256입니다.)

In [10]:
# ... (self.features 이후)

        # Classifier 정의
        self.classifier = nn.Sequential(
            # [문제 1] 텐서를 1차원으로 평탄화 (Flatten)
            nn.Flatten(),

            # [문제 2] Fully Connected Layer (입력: 4096, 출력: 256)
            nn.Linear(64*8*8, 256),

            nn.ReLU(),
            nn.Linear(256, 10)  # CIFAR-10 클래스 10개
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 15)

문항 6. 모델 평가 함수 구현 모델 평가(evaluate) 함수에서, 학습 모드(Dropout, Batch Normalization 등)를 비활성화하고 기울기(gradient) 계산을 중단하도록 하는 코드를 [문제] 위치에 각각 작성하시오.

In [8]:
def evaluate(model, loader):
    # [문제 1] 모델을 평가 모드(evaluation mode)로 설정
    model.eval()

    correct = 0
    total = 0

    # [문제 2] 기울기 계산을 중단하는 컨텍스트 매니저
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

자율코딩

In [ ]:
layer_outputs = OrderedDict()

def register_hooks(model):
      """
    모델의 모든 레이어에 forward hook을 등록

    Args:
        model: PyTorch 모델

    Returns:
        list: hook handle 리스트 (나중에 제거용)
    """
    handles = []

    def hook_fn(module, input, output):
        """
        Forward hook 함수

        Args:
            module: 현재 레이어 객체
            input: 레이어 입력 (튜플)
            output: 레이어 출력 (텐서)
        """
        layer_name =  module.__class__.__name__
        count = sum(1 for k in layer_outputs.keys() if layer_name in k)
        if count > 0:
          layer_name = f"{layer_name}_{count+1}"
        layer_outputs[layer_name] = output.shape
    for name, module in model.named_modules():
      if len(list(module.children())) == 0 and module != model:
        handle = module.register_forward_hook(hook_fn)
        handles.append(handle)
    return handles

print('모든 레이어에 Hook 등록 중...')
hook_handles = register_hooks(model)
print(f'총 {len(hook_handles)}개의 Hook이 등록되었습니다.\n')

dummy_input = torch.randn(1, 3, 32, 32).to(device)

print('더미 입력으로 순전파 실행 중...')
with torch.no_grad():
  output = model(dummy_input)
print('\n순전파 완료! 각 레이어의 출력 크기:\n')

print('='*60)
print(f'{"레이어 이름":<25} {"출력 Shape":>30}')
print('-'*60)
print(f'{"입력 이미지":<25} {str(dummy_input.shape):>30}')
print('-'*60)

for layer_name, shape in layer_outputs,items():
  print(f'{layer_name:<25} {str(tuple(shape)):>30}')

print('='*60)

print('\nHook 제거 중...')
for handle in hook_handles:
  handle.remove()
print('모든 Hook이 제거되었습니다.')

print('\n출력 크기 해석:')
print('  - 형식: [배치 크기, 채널 수, 높이, 너비]')
print('  - Conv 후: 채널 수 증가, 공간 크기는 padding으로 유지')
print('  - MaxPool 후: 채널 수 유지, 공간 크기 절반으로 축소')
print('  - Flatten 후: [배치 크기, 총 특징 수]로 1차원화')
print('  - FC 후: [배치 크기, 출력 뉴런 수]')
